In [ ]:
import gzip # For reading compressed .ptt.gz files
import os

# Parse a .ptt.gz file and extract gene info (start, end, strand)
def parse_ptt(ptt_path):
    with gzip.open(ptt_path, 'rt') as f:
        genes = []
        for line in f.readlines()[3:]:  # Skip 3 header lines
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                try:
                    # Extract start and end from the location string
                    start, end = map(int, parts[0].split('..'))
                    strand = parts[1]
                    genes.append((start, end, strand))
                except:
                    continue
    return sorted(genes, key=lambda x: x[0]) # Sort genes by start position

# Predict operons from gene list
def predict_operons(genes):
    operons, op = [], [genes[0]] # Start first operon with the first gene
    for curr in genes[1:]:
        prev = op[-1]
        # Check co-directional and intergenic distance
        if curr[2] == prev[2] and (curr[0] - prev[1]) <= 50:
            op.append(curr) # Add to current operon
        else:
            operons.append(op) # Save current operon
            op = [curr] # Start new operon 
    operons.append(op) # Append last operon
    return operons

def write_operons(operons, out_file):
    with open(out_file, 'w') as f:
        for i, op in enumerate(operons, 1):
            f.write(f"Operon {i} ({len(op)} genes):\n")
            for g in op:
                start, end, strand = g
                f.write(f"  Start: {start}, End: {end}, Strand: {strand}\n")
            f.write("\n")

# Run on PTT files
ptt_files = [
    "E_coli_K12_MG1655.ptt.gz",
    "B_subtilis_168.ptt.gz",
    "Halobacterium_NRC1.ptt.gz",
    "Synechocystis_PCC6803_uid159873.ptt.gz"
]

for file in ptt_files:
    print(f"Processing: {file}")
    genes = parse_ptt(file)   # Extract gene list
    operons = predict_operons(genes) # Predict operons
    out_file = file.replace(".ptt.gz", "_operons.txt")
    write_operons(operons, out_file) # Result
    print(f" Total genes: {len(genes)}")
    print(f" Predicted operons: {len(operons)}")
    print(f" Saved to: {out_file}\n")


Processing: E_coli_K12_MG1655.ptt.gz
 Total genes: 4146
 Predicted operons: 2660
 Saved to: E_coli_K12_MG1655_operons.txt

Processing: B_subtilis_168.ptt.gz
 Total genes: 4176
 Predicted operons: 2653
 Saved to: B_subtilis_168_operons.txt

Processing: Halobacterium_NRC1.ptt.gz
 Total genes: 2075
 Predicted operons: 1454
 Saved to: Halobacterium_NRC1_operons.txt

Processing: Synechocystis_PCC6803_uid159873.ptt.gz
 Total genes: 3170
 Predicted operons: 2507
 Saved to: Synechocystis_PCC6803_uid159873_operons.txt



In [1]:
# Parse the GFF file and extract gene information
def parse_gff(gff_path):
    contigs = {}
    with open(gff_path, 'r') as f:
        for line in f:
            if line.startswith("#"): continue  
            parts = line.strip().split('\t')
            if len(parts) < 9: continue  

            try:
                contig = parts[0]
                start = int(parts[3])     # Gene start
                end = int(parts[4])       # Gene end
                strand = parts[6]         # Strand: '+' or '-'
                attributes = parts[8]     # Last column with gene ID info

                # Extract gene ID from attributes 
                gene_id = "unknown"
                for key in ["ID=", "locus_tag=", "Name="]:
                    if key in attributes:
                        gene_id = attributes.split(key)[1].split(';')[0]
                        break

                # Store gene info in a dictionary
                gene = {"start": start, "end": end, "strand": strand, "id": gene_id}
                contigs.setdefault(contig, []).append(gene)
            except:
                continue  # Skip lines with conversion errors

    # Sort genes on each contig by start coordinate
    for c in contigs:
        contigs[c].sort(key=lambda x: x["start"])
    return contigs

# Predict operons from sorted gene list
def predict_operons(genes):
    operons, op = [], [genes[0]]  # Start first operon
    for curr in genes[1:]:
        prev = op[-1]
        # Check if gene is co-directional and within 50 bp
        if curr["strand"] == prev["strand"] and (curr["start"] - prev["end"]) <= 50:
            op.append(curr)
        else:
            operons.append(op)  # Store current operon
            op = [curr]         # Start new operon
    operons.append(op)  # Add the last operon
    return operons

# Write operon predictions to a text file
def write_operons(contig_operons, out_file):
    with open(out_file, 'w') as f:
        for contig, ops in contig_operons.items():
            f.write(f"Contig: {contig}\n")
            for i, op in enumerate(ops, 1):
                f.write(f"  Operon {i} ({len(op)} genes):\n")
                for g in op:
                    f.write(f"    Gene ID: {g['id']}, Start: {g['start']}, End: {g['end']}, Strand: {g['strand']}\n")
            f.write("\n")

gff_file = "2088090036.gff"  # Input GFF file
contigs = parse_gff(gff_file)  # Parse and group genes by contig

# Predict operons on each contig
contig_operons = {c: predict_operons(genes) for c, genes in contigs.items()}

# Write results to file
output_file = "metagenome_operons.txt"
write_operons(contig_operons, output_file)
print(f"Operon data written to: {output_file}")


Operon data written to: metagenome_operons.txt
